In [1]:
import pandas as pd
from pathlib import Path
import os 
from thefuzz import process
import pandas as pd
import os
import re
import tkinter as tk
from tkinter import filedialog
from classes.clase_reportes_new import ReportClassNew
from classes.send_mail import MailSender
import requests
from difflib import get_close_matches
import unicodedata
rc = ReportClassNew()
ms = MailSender()


In [2]:
import pandas as pd
from pathlib import Path
import re

## Correos de contabilidad
## Cambiar ruta de archivo y origen de pdfs
# df = pd.read_excel(r'D:\Desktop\prueba_correos.xlsx')
df = pd.read_excel(r'd:\Downloads\Base de datos.xlsx', sheet_name='BASE DE DATOS DIRECTOS PCN')
df = df[['CEDULA', 'CORREO CORPORATIVO', 'CORREO']]
df = df[df['CEDULA'].notnull()]


In [3]:
df

,CEDULA,CORREO CORPORATIVO,CORREO
0,1001052406,acuellar@lapocion.com,aixaceleste9@gmail.com
1,1113673258,agomez@lapocion.com,ale-0395@hotmail.com
2,1006130891,acalderon@lapocion.com,andresfelipecalderon33@hotmail.com
3,1130587984,andresvasquez@lapocion.com,andresfvb24@hotmail.com
4,1130629411,mcolorado@lapocion.com,andresmauriciocolaroda@gmai.com
...,...,...,...
104,1006099505,NaN,msanchezmontoya02@gmail.com
105,1193303493,NaN,yarmyyuliana@gmail.com
106,1107068961,NaN,wilson.leon@correounivalle.edu.co
107,1111545220,NaN,valentina.marquez0414@gmail.com


In [4]:
df['CEDULA'] = df['CEDULA'].astype(int)
df['CORREO CORPORATIVO'] = df['CORREO CORPORATIVO'].fillna(df['CORREO'])
# Configuración de ruta
ruta_consolidado = Path(r"G:\.shortcut-targets-by-id\1wo7Yn8Q9Ege2UAGn8-TjvMD5YFQLC-Mk\CERTIFICADOS DE INGRESOS 2025")

html_formato = """
<!DOCTYPE html>
<html lang="es">
<head>
    <meta charset="UTF-8">
</head>
<body style="font-family: 'Segoe UI', Arial, sans-serif; background-color: #f4f7f6; margin: 0; padding: 20px;">
    <div style="max-width: 600px; margin: 0 auto; background-color: #ffffff; border-radius: 8px; overflow: hidden; border: 1px solid #e0e0e0; box-shadow: 0 4px 6px rgba(0,0,0,0.05);">

        <div style="background-color: #2c3e50; color: #ffffff; padding: 20px; text-align: center;">
            <h2 style="margin: 0; font-size: 24px; letter-spacing: 1px;">La Poción</h2>
            <p style="margin: 5px 0 0 0; font-size: 14px; opacity: 0.8;">Área de Recursos Humanos</p>
        </div>

        <div style="padding: 30px; color: #333333; line-height: 1.6;">
            <h3 style="color: #2c3e50; margin-top: 0; border-bottom: 2px solid #eee; padding-bottom: 10px;">
                Certificado de Ingresos y Retenciones 2025
            </h3>

            <p>Estimado(a) colaborador(a),</p>

            <p>Esperamos que te encuentres muy bien.</p>

            <p>Por medio del presente correo hacemos entrega del <strong>Certificado de Ingresos y Retención en la Fuente correspondiente al año gravable 2025</strong>.</p>

            <div style="background-color: #e8f4f8; padding: 15px; border-left: 4px solid #2980b9; border-radius: 4px; margin: 25px 0;">
                <p style="margin: 0;">
                    📄 <strong>Documento adjunto:</strong> Te sugerimos descargarlo y guardarlo para tus trámites personales y tributarios (como la declaración de renta).
                </p>
            </div>

            <p>Si  tienes alguna inquietud, con gusto sera atendida.</p>

            <p style="margin-top: 30px;">Cordialmente,</p>
            <p style="margin-bottom: 0;"><strong>El equipo de Recursos Humanos</strong><br>La Poción</p>
        </div>

        <div style="background-color: #f9f9f9; padding: 15px 30px; text-align: center; border-top: 1px solid #eeeeee;">
            <p style="margin: 0; font-size: 11px; color: #888888; line-height: 1.4;">
                Este es un mensaje generado automáticamente. Te recordamos que el documento adjunto contiene información confidencial; por favor, manéjalo con la debida precaución.
            </p>
        </div>

    </div>
</body>
</html>
"""

# ==============================================================================
# PASO 1: Escanear la carpeta UNA SOLA VEZ y mapear archivos por cédula exacta
# ==============================================================================
diccionario_certificados = {}
archivos_en_carpeta = set()  # Guardará los nombres de todos los PDFs para el control final

if ruta_consolidado.exists():
    for archivo in ruta_consolidado.iterdir():
        if archivo.is_file() and archivo.suffix.lower() == '.pdf':
            match = re.match(r'^(\d+)', archivo.stem)
            if match:
                cedula_del_archivo = match.group(1)
                diccionario_certificados[cedula_del_archivo] = archivo
                archivos_en_carpeta.add(archivo.name)  # Registramos el archivo
    print(f"--> Mapeo completado. Se indexaron {len(diccionario_certificados)} archivos PDF.\n")
else:
    print(f"Error: La ruta {ruta_consolidado} no existe.")
    exit()

# Listas de control para el reporte final
enviados = []
sin_adjunto = []
sin_correo_valido = []
archivos_enviados = set()

# ==============================================================================
# PASO 2: Iterar el Excel y procesar los envíos
# ==============================================================================
print("=== INICIANDO PROCESAMIENTO DE EXCEL ===")
for index, fila in df.iterrows():
    cedula = str(fila['CEDULA']).strip()
    correo_cliente = str(fila['CORREO CORPORATIVO']).strip()

    # Validación 1: Verificar si tiene correo válido escrito en el Excel
    if correo_cliente.upper() in ['NO TIENE CORREO', 'NAN', ''] or not correo_cliente:
        sin_correo_valido.append(f"Cédula {cedula} (Valor en Excel: '{correo_cliente}')")
        continue

    # Búsqueda por coincidencia exacta en el diccionario
    if cedula in diccionario_certificados:
        archivo_exacto = diccionario_certificados[cedula]
        
        print(f"[OK] Coincidencia: Correo para {correo_cliente} con archivo {archivo_exacto.name}")
        
        # Guardamos en el registro de enviados
        enviados.append(f"Cédula {cedula} -> {correo_cliente} (Archivo: {archivo_exacto.name})")
        archivos_enviados.add(archivo_exacto.name)  # Marcamos este archivo como "procesado"
        
        # Descomenta las líneas de abajo cuando estés listo para el envío real:
        ms.enviar_correo(
            destinatarios=correo_cliente,
            asunto=f"Certificado de Retenciones Año Gravable 2025 {cedula}",
            # cc=['mafricano@lapocion.com', 'gisselquintero@lapocion.com'],
            cuerpo_html=html_formato,
            adjuntos=[str(archivo_exacto)]
        )

    else:
        # Validación 2: Si está en el Excel pero NO tiene PDF, lo registramos para avisar
        sin_adjunto.append(f"Cédula {cedula} -> Correo: {correo_cliente}")

print("\n=== PROCESAMIENTO TERMINADO ===\n")

# ==============================================================================
# PASO 3: Validación cruzada e Informe de Control
# ==============================================================================
# Calculamos qué archivos se quedaron en la carpeta sin ser enviados a nadie
archivos_huérfanos = archivos_en_carpeta - archivos_enviados

print("==========================================================================")
print("                       REPORTE DE CONTROL DE ENVÍOS                      ")
print("==========================================================================")
print(f"Total de correos preparados/enviados exitosamente: {len(enviados)}")

print(f"\n1. PERSONAS EN EXCEL QUE NO TIENEN ARCHIVO PDF ADJUNTO ({len(sin_adjunto)}):")
if sin_adjunto:
    for item in sin_adjunto:
        print(f"   - {item}")
else:
    print("   (Ninguno. Todos los registrados en el Excel con correo tenían su PDF correspondiente).")

print(f"\n2. ARCHIVOS PDF EN CARPETA QUE NO SE ENVIARON ({len(archivos_huérfanos)}):")
print("   *Nota: Estas cédulas están en los archivos físicos pero NO aparecen en tu Excel*")
if archivos_huérfanos:
    for archivo_nombre in sorted(archivos_huérfanos):
        print(f"   - [No Enviado] {archivo_nombre}")
else:
    print("   (Ninguno. ¡Todos los archivos PDF de la carpeta fueron asignados y enviados!).")

print(f"\n3. REGISTROS OMITIDOS POR NO TENER UN CORREO VÁLIDO EN EXCEL ({len(sin_correo_valido)}):")
if sin_correo_valido:
    for item in sin_correo_valido:
        print(f"   - {item}")
else:
    print("   (Ninguno. Todos los registros del Excel tenían texto en la columna de correo).")
print("==========================================================================")

--> Mapeo completado. Se indexaron 93 archivos PDF.

=== INICIANDO PROCESAMIENTO DE EXCEL ===
[OK] Coincidencia: Correo para acuellar@lapocion.com con archivo 1001052406 Certificado de ingresos 2025.pdf
📨 Enviando correo a: acuellar@lapocion.com
✅ Correo enviado correctamente.
[OK] Coincidencia: Correo para agomez@lapocion.com con archivo 1113673258 Certificado de ingresos 2025.pdf
📨 Enviando correo a: agomez@lapocion.com
✅ Correo enviado correctamente.
[OK] Coincidencia: Correo para acalderon@lapocion.com con archivo 1006130891 Certificado de ingresos 2025.pdf
📨 Enviando correo a: acalderon@lapocion.com
✅ Correo enviado correctamente.
[OK] Coincidencia: Correo para andresvasquez@lapocion.com con archivo 1130587984 Certificado de ingresos 2025.pdf
📨 Enviando correo a: andresvasquez@lapocion.com
✅ Correo enviado correctamente.
[OK] Coincidencia: Correo para mcolorado@lapocion.com con archivo 1130629411 Certificado de ingresos 2025.pdf
📨 Enviando correo a: mcolorado@lapocion.com
✅ Correo